# Results KPI Monitor

Purpose: fast daily health check (PASS/WARN) from latest scorecard artifacts.

This is the morning checkpoint, not the root-cause notebook.

Fast daily scan for model health, WARN state, and operator action.

In [1]:
from pathlib import Path
import polars as pl
from IPython.display import HTML, display

ROOT = Path.cwd().resolve()
for candidate in [ROOT, *ROOT.parents]:
    if (candidate / "production").exists() and (candidate / "src" / "Python").exists():
        ROOT = candidate
        break

ODDS_DIR = ROOT / "artifacts" / "odds_log"
SCORECARD_PATH = ODDS_DIR / "model_health_scorecard_daily.parquet"
GATE_PATH = ODDS_DIR / "gate_next_n_comparison.parquet"


def show_table(df: pl.DataFrame, max_rows: int = 30, height: int = 420):
    pdf = df.to_pandas()
    if len(pdf) <= max_rows:
        display(pdf)
        return
    table = pdf.to_html(index=False, na_rep="—")
    display(HTML(f"<div style='max-height:{height}px; overflow:auto; border:1px solid #4443; border-radius:6px'>{table}</div>"))


print("repo:", ROOT)
print("scorecard:", SCORECARD_PATH)
print("gate:", GATE_PATH)

repo: C:\Users\ckaplinger\Downloads\Personal-Projects\MLB-Props
scorecard: C:\Users\ckaplinger\Downloads\Personal-Projects\MLB-Props\artifacts\odds_log\model_health_scorecard_daily.parquet
gate: C:\Users\ckaplinger\Downloads\Personal-Projects\MLB-Props\artifacts\odds_log\gate_next_n_comparison.parquet


In [2]:
if not SCORECARD_PATH.exists():
    print("Missing scorecard artifact. Run results dashboard scorecard section first.")
else:
    score = pl.read_parquet(SCORECARD_PATH).sort("snapshot_utc")
    latest = score.tail(1)
    cols = [
        c
        for c in [
            "snapshot_utc",
            "n_joined",
            "n_warn",
            "mae_err_k_rate",
            "under_bias_tbf",
            "worst_matchup_tier_mae_err_k_rate",
            "long_rest_bias_tbf",
        ]
        if c in latest.columns
    ]
    print("latest scorecard row")
    show_table(latest.select(cols))
    print("\nrecent trend (last 10)")
    trend_cols = [c for c in ["snapshot_utc", "n_warn", "mae_err_k_rate", "under_bias_tbf"] if c in score.columns]
    show_table(score.select(trend_cols).tail(10))

latest scorecard row


,snapshot_utc,n_joined,n_warn,mae_err_k_rate,under_bias_tbf,long_rest_bias_tbf
0,2026-08-11T14:58:57.251535+00:00,254,3,0.078657,-1.433441,-7.523606



recent trend (last 10)


,snapshot_utc,n_warn,mae_err_k_rate,under_bias_tbf
0,2026-08-07T17:54:23.636635+00:00,3,0.073447,-2.516463
1,2026-08-10T16:12:47.313541+00:00,4,0.080804,-1.532461
2,2026-08-11T14:58:57.251535+00:00,3,0.078657,-1.433441


In [3]:
if not GATE_PATH.exists():
    print("No gate comparison artifact yet.")
else:
    gate = pl.read_parquet(GATE_PATH).sort("snapshot_utc")
    cols = [c for c in ["snapshot_utc", "next_n", "gate_pnl_delta", "gate_clv_delta_pp", "gate_bet_count_delta"] if c in gate.columns]
    show_table(gate.select(cols).tail(10))

,snapshot_utc,gate_pnl_delta
0,2026-08-10T15:47:46.483637+00:00,0.0
1,2026-08-11T14:03:25.906891+00:00,0.0
2,2026-08-11T14:04:37.249717+00:00,0.0
3,2026-08-12T14:06:12.450967+00:00,0.0
4,2026-08-12T14:08:19.653028+00:00,0.0
